# Back translation from Odia to German

Here is a complete, well-commented Python script to perform back-translation using Meta's NLLB model.

The primary goal of back-translation is to create a large, synthetic parallel dataset. The most crucial part, which this script accomplishes, is translating your monolingual Odia corpus into German. This creates (Real Odia -> Synthetic German) pairs, massively increasing your training data.

This script is designed to be efficient by processing sentences in batches, which is much faster than translating one by one, especially if you have a GPU.

In [ ]:
!pip install torch transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from transformers import pipeline
import os
from tqdm import tqdm

In [ ]:
# --- 1. CONFIGURATION ---
# --- Input and Output Files ---
MONOLINGUAL_ODIA_FILE = "/content/drive/MyDrive/Thesis/test/data/raw/authentic_odia_corpus_poc_2.txt"
SYNTHETIC_GERMAN_FILE = "/content/drive/MyDrive/Thesis/test/data/raw/synthetic_nllb-200-distilled-600M_german_corpus_poc_2.txt"

# --- Model and Language Configuration ---
# We use a distilled (smaller, faster) version of NLLB. It offers a great balance of speed and quality.
MODEL_NAME = "facebook/nllb-200-3.3B"
# MODEL_NAME = "facebook/nllb-200-distilled-600M"
ODIA_LANG_CODE = "ory_Orya"     # The language code for Odia in NLLB
GERMAN_LANG_CODE = "deu_Latn"   # The language code for German in NLLB

# --- Performance Configuration ---
# Process sentences in batches for significant speedup, especially on GPU.
# Adjust based on your available RAM/VRAM. Start with 8 or 16.
BATCH_SIZE = 16

def create_synthetic_dataset():
  """
  Main function to perform the Odia to German translation.
  """
  # --- Preliminary Check ---
  if not os.path.exists(MONOLINGUAL_ODIA_FILE):
    print(f"--- ERROR: Input file '{MONOLINGUAL_ODIA_FILE}' not found. ---")
    print("Please create this file and populate it with your monolingual Odia sentences.")
    # Create a dummy file for demonstration purposes
    with open(MONOLINGUAL_ODIA_FILE, "w", encoding="utf-8") as f:
      f.write("ଏହି ଯୋଜନାର ମୁଖ୍ୟ ଉଦ୍ଦେଶ୍ୟଗୁଡ଼ିକ ହେଲା।\n")
      f.write("ପ୍ରତ୍ୟେକ ଘରକୁ ବିଦ୍ୟୁତ ସଂଯୋଗ ଦେବା।\n")
    print(f"A dummy '{MONOLINGUAL_ODIA_FILE}' has been created so you can test the script.")
    return

  # --- 2. SETUP THE MODEL ---
  # Set up the device: Use GPU if available (device=0), otherwise CPU (device=-1)
  device = 0 if torch.cuda.is_available() else -1
  if device == 0:
    print("✅ GPU detected. Using GPU for high-speed translation.")
  else:
    print("⚠️ No GPU detected. Using CPU for translation (this will be much slower).")

  # Initialize the Hugging Face translation pipeline.
  # The model (a few GBs) will be downloaded automatically the first time you run this.
  print(f"Loading NLLB model '{MODEL_NAME}'... This may take a while on the first run.")
  translator = pipeline(
      "translation",
      model=MODEL_NAME,
      src_lang=ODIA_LANG_CODE,
      tgt_lang=GERMAN_LANG_CODE,
      device=device
  )

  # --- 3. READ THE DATA ---
  print(f"Reading Odia sentences from '{MONOLINGUAL_ODIA_FILE}'...")
  with open(MONOLINGUAL_ODIA_FILE, 'r', encoding='utf-8') as f:
    # Read all lines, stripping any leading/trailing whitespace
    odia_sentences = [line.strip() for line in f if line.strip()]

  if not odia_sentences:
    print("--- ERROR: The input file is empty. Please add Odia sentences to it. ---")
    return

  print(f"Found {len(odia_sentences)} sentences to translate.")

  # --- 4. TRANSLATE AND SAVE ---
  print(f"Starting translation in batches of {BATCH_SIZE}. This may take a long time for a large corpus...")
  # Collect all translated sentences
  translated_sentences = []
  for i in tqdm(range(0, len(odia_sentences), BATCH_SIZE), desc="Translating Batches"):
    # Create a batch of sentences
    batch = odia_sentences[i:i + BATCH_SIZE]
    # Translate the entire batch
    translated_batch = translator(batch, max_length=1024)
    # Collect translated sentences
    translated_sentences.extend([translation['translation_text'] for translation in translated_batch])

  # Write translated sentences to the output file
  with open(SYNTHETIC_GERMAN_FILE, 'w', encoding='utf-8') as f_out:
    # Write sentences, adding \n\n between sentences but not after the last one
    for i, german_sentence in enumerate(translated_sentences):
      if i < len(translated_sentences) - 1:
        f_out.write(german_sentence + '\n\n')  # Add blank line between sentences
      else:
        f_out.write(german_sentence)  # No trailing blank line for the last sentence

  print(f"\n✅ Translation complete! Synthetic German corpus saved in '{SYNTHETIC_GERMAN_FILE}'.")

# --- Step 5: RUN THE FUNCTION ---
create_synthetic_dataset()

✅ GPU detected. Using GPU for high-speed translation.
Loading NLLB model 'facebook/nllb-200-3.3B'... This may take a while on the first run.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

Device set to use cuda:0


Reading Odia sentences from '/content/drive/MyDrive/Thesis/test/data/raw/authentic_odia_corpus_poc_2.txt'...
Found 2461 sentences to translate.
Starting translation in batches of 16. This may take a long time for a large corpus...


Translating Batches: 100%|██████████| 154/154 [1:39:27<00:00, 38.75s/it]


✅ Translation complete! Synthetic German corpus saved in '/content/drive/MyDrive/Thesis/test/data/raw/synthetic_nllb-200-distilled-600M_german_corpus_poc_2.txt'.
